In [1]:
import torch
from transformers import BertTokenizer, BertModel
from triton.testing import do_bench

/Users/minhhuunguyen/REPOSITORY/minhhuunguyen.github.io/posts/blog-sharing/1_pixta_seminar/2_pytorch2_seminar/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.0
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ModuleNotFoundError: No module named 'triton'

In [ ]:
torch.__version__

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
def run_benchmark(fn):
    exec_time, prctl20, prctl80 = do_bench(fn, warmup=100, rep=1000)
    return exec_time

In [ ]:
def cal_improved_percent(exec_time, opt_exec_time):
    avg_exec_time = (sum(exec_time) / len(exec_time))
    avg_opt_exec_time = (sum(opt_exec_time) / len(opt_exec_time))
    
    print('avg_exec_time', avg_exec_time)
    print('avg_opt_exec_time', avg_opt_exec_time)

    return f'{round((avg_exec_time - avg_opt_exec_time) / avg_opt_exec_time * 100, 2)}%'

# 1. Test ResNet

In [ ]:
def run_batch(model, optimizer, device):
    x = torch.randn(32, 3, 64, 64).to(device)
    optimizer.zero_grad()
    out = model(x)
    out.sum().backward()
    optimizer.step()

In [ ]:
def run_only_forward(model, device):
    x = torch.randn(32, 3, 64, 64).to(device)
    out = model(x)

In [ ]:
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [ ]:
# device = 'cpu'
device = 'cuda:0'
NUM_LOOP = 1

In [ ]:
exec_time = [
    run_benchmark(lambda: run_batch(model.to(device), optimizer, device))
    for _ in range(NUM_LOOP)
]

In [ ]:
torch._dynamo.reset() # Reset before compile again
opt_model_default = torch.compile(model)

opt_exec_time_default = [
    run_benchmark(lambda: run_batch(opt_model_default.to(device), optimizer, device))
    for _ in range(NUM_LOOP)
]

cal_improved_percent(exec_time, opt_exec_time_default)

In [ ]:
torch._dynamo.reset() # Reset before compile again

opt_model_max = torch.compile(model, mode='max-autotune')

opt_exec_time_max = [
    run_benchmark(lambda: run_batch(opt_model_max.to(device), optimizer, device))
    for _ in range(NUM_LOOP)
]

cal_improved_percent(exec_time, opt_exec_time_max)

In [ ]:
exec_time = [
    run_benchmark(lambda: run_only_forward(model.to(device), device))
    for _ in range(NUM_LOOP)
]

In [ ]:
torch._dynamo.reset() # Reset before compile again

opt_model_default = torch.compile(model)

opt_exec_time_default = [
    run_benchmark(lambda: run_only_forward(opt_model_default.to(device), device))
    for _ in range(NUM_LOOP)
]

cal_improved_percent(exec_time, opt_exec_time_default)

In [ ]:
torch._dynamo.reset() # Reset before compile again

opt_model_max = torch.compile(model, mode='max-autotune')

opt_exec_time_max = [
    run_benchmark(lambda: run_only_forward(opt_model_max.to(device), device))
    for _ in range(NUM_LOOP)
]

cal_improved_percent(exec_time, opt_exec_time_max)

# 2. Test Transformers

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
input_ = tokenizer('This is a text to test the model', return_tensors='pt')

In [ ]:
model = BertModel.from_pretrained("bert-base-uncased")

torch._dynamo.reset() # Reset before compile again

opt_model = torch.compile(model, mode='max-autotune')

In [ ]:
def run_foward_transformers(model, input_):
    output = model(**input_)

In [ ]:
exec_time = [
    run_benchmark(lambda: run_foward_transformers(model.to(device), input_.to(device)))
    for _ in range(NUM_LOOP)
]

In [ ]:
opt_exec_time_max = [
    run_benchmark(lambda: run_foward_transformers(opt_model.to(device), input_.to(device)))
    for _ in range(NUM_LOOP)
]

cal_improved_percent(exec_time, opt_exec_time_max)